# Phase 1 — Khảo sát dữ liệu FloodNet

Notebook này **mỏng**: nó chỉ gọi `src/floodcount/data/audit.py` trong repo, không
chứa logic. Sửa logic thì sửa file `.py` rồi `git pull`, không sửa trong notebook —
để code còn tái lập được và không bị lệch giữa các phiên.

**Không cần GPU cho Phase 1.** Bước này chỉ đọc ảnh, đếm pixel và tách component
bằng CPU. Chạy ở runtime CPU cũng được, khỏi tốn suất T4.

**Không cần cài MMDetection.** Bước này chỉ dùng `cv2`, `numpy`, `yaml`. Đừng chạy
notebook `00_colab_setup.ipynb` chỉ để làm việc này — tốn 10 phút vô ích.

## ⚠️ Chạy lần lượt từ trên xuống, bắt đầu từ ô `[1.1] CẤU HÌNH`

Colab chạy **từng ô độc lập**, không tự chạy ô phía trên. Nhảy thẳng xuống ô `[1.5]`
hoặc `[1.6]` sẽ lỗi `NameError: name 'EDA_OUT' is not defined` — nghĩa là ô `[1.1]`
chưa chạy, không phải lỗi code. Ba ô chạy đều đã có sẵn thông báo nhắc việc này.


In [ ]:
# [1.1] CẤU HÌNH — sửa ở đây, không sửa trong code
REPO_URL  = "https://github.com/21xDHoang/flood-house-counting.git"
REPO_DIR  = "/content/flood-house-counting"

DRIVE_DIR = "/content/drive/MyDrive/Flood_House_AI"
ZIP_PATH  = f"{DRIVE_DIR}/floodnet_raw.zip"      # 13 GB, đã có sẵn trên Drive
EDA_OUT   = f"{DRIVE_DIR}/runs/eda"              # nơi ghi báo cáo + ảnh overlay

# Thư mục trung gian cho lần CHẠY THỬ, tách riêng để không lẫn vào kết quả thật
WORK_DIR_THU = "/content/floodnet_work_thu"

print("Đã nạp cấu hình Phase 1.")
print(f"  repo   : {REPO_URL}")
print(f"  zip    : {ZIP_PATH}")
print(f"  kết quả: {EDA_OUT}")

In [ ]:
# [1.2] GẮN GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# [1.3] LẤY CODE TỪ GITHUB
#
# /content mất sạch mỗi lần runtime reset (xem docs/NOTES.md muc 1.6), nên phiên
# nào cũng phải làm bước này. Có sẵn repo rồi thì `git pull` cho nhanh.
import os
import subprocess


def chay(lenh, **them):
    kq = subprocess.run(lenh, capture_output=True, text=True, **them)
    if kq.stdout.strip():
        print(kq.stdout.strip())
    if kq.returncode != 0:
        print("LỖI:", kq.stderr.strip())
    return kq.returncode


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Đã có repo, cập nhật code mới nhất:")
    chay(["git", "-C", REPO_DIR, "pull"])
else:
    print("Chưa có repo, clone về:")
    chay(["git", "clone", REPO_URL, REPO_DIR])

print()
print("Commit đang dùng:")
chay(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])

# Kiểm repo đã có đủ code Phase 1 chưa. Clone từ lúc chưa push sẽ thiếu configs/,
# và audit.py khi đó thoát ngay mà không in gì — rất khó đoán ra.
print()
print("Kiểm tra code Phase 1:")
for duong_dan_can in ("configs/data.yaml", "scripts/audit.py",
                      "src/floodcount/data/audit.py"):
    co = os.path.exists(os.path.join(REPO_DIR, duong_dan_can))
    print(f"  [{'OK ' if co else 'THIẾU'}] {duong_dan_can}")
    if not co:
        print("  -> Thiếu code. Xoá thư mục repo rồi chạy lại ô này để clone lại.")
        break

In [ ]:
# [1.4] KIỂM TRA THƯ VIỆN CẦN THIẾT (nhẹ, KHÔNG cài MMDetection)
import importlib
import sys

for ten_module, ten_goi_pip in [("cv2", "opencv-python"),
                                ("yaml", "pyyaml"),
                                ("numpy", "numpy")]:
    try:
        m = importlib.import_module(ten_module)
        print(f"{ten_module:<6}: {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{ten_module:<6}: THIẾU -> đang cài {ten_goi_pip} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", ten_goi_pip])

print("\nSẵn sàng.")

---
## Bước 1 — Chạy thử 3 ảnh mỗi split

**Chạy ô này trước, đừng chạy thẳng ô đầy đủ.**

Mục đích: phát hiện sớm nếu cấu trúc zip khác kỳ vọng. Chạy thử mất ~30 giây; nếu
cấu trúc sai mà chạy thẳng thì mất ~20 phút mới biết.

Đọc kỹ phần **CẤU TRÚC ZIP** và **CẢNH BÁO** ở đầu kết quả.


In [ ]:
# [1.5] CHẠY THỬ — 3 ảnh mỗi split
#
# Dùng subprocess chứ không dùng `!` của Colab vì hai lý do:
#   - Code là Python thuần nên kiểm tra được cú pháp trên máy, không phải mở Colab
#     mới biết sai.
#   - Không truyền capture_output: tiến trình con in thẳng ra màn hình nên log
#     hiện dần theo thời gian thực, không bị dồn lại tới lúc chạy xong.
#
# `stderr=subprocess.STDOUT` là BẮT BUỘC, không phải cho đẹp: Colab chỉ nối
# fd 1 của tiến trình con vào ô output, còn fd 2 thì đi vào log của máy chủ
# Jupyter — người dùng KHÔNG nhìn thấy. Thiếu dòng này thì mọi traceback đều
# biến mất, chỉ còn "Mã thoát: 1" trống trơn, không biết lỗi gì (đã gặp thật).
#
# Phải tự kiểm ở TỪNG ô, không viết thành hàm dùng chung: nếu ô [1.1] chưa chạy
# thì hàm định nghĩa trong đó cũng không tồn tại, lại quay về NameError khó hiểu.
if "EDA_OUT" not in globals():
    raise SystemExit("Chưa chạy ô [1.1] CẤU HÌNH. "
                     "Hãy chạy lần lượt từ ô [1.1] trở xuống rồi thử lại.")

import subprocess
import sys

lenh_thu = [sys.executable, "scripts/audit.py",
            "--config", "configs/data.yaml",
            "--max-images", "3",
            "--work-dir", WORK_DIR_THU,
            "--output-dir", EDA_OUT + "_thu"]
print("Chạy: " + " ".join(lenh_thu))
print("=" * 72)
kq = subprocess.run(lenh_thu, cwd=REPO_DIR, stderr=subprocess.STDOUT)
print("=" * 72)
print("Mã thoát:", kq.returncode, "(0 là thành công)")

**Đọc kết quả chạy thử:**

- Phần `CẤU TRÚC ZIP` phải liệt kê được các thư mục `train/`, `val/`, `test/`.
- Dòng có `[ mask]` là thư mục nhãn thật; dòng `[ anh]` là ảnh màu.
- Nếu thấy cảnh báo `bẫy ColorMasks` thì **đúng như dự kiến** — script đã tự loại
  thư mục đó ra, không dùng nó làm nhãn.
- Nếu báo `Không ghép được cặp ảnh/mask nào` thì **dừng lại**, gửi nguyên phần
  `CẤU TRÚC ZIP` cho Claude.

Chạy thử xong mà ổn thì mới chạy ô tiếp theo.


---
## Bước 2 — Chạy đầy đủ

Toàn bộ train + val + test, tách component cho từng ảnh. Mất khoảng **15–30 phút**
tuỳ số ảnh thật.

Ô này **tự chạy tiếp nếu bị ngắt**: kết quả từng ảnh được ghi ngay xuống đĩa, chạy
lại sẽ bỏ qua ảnh đã xử lý. Colab ngắt giữa chừng thì cứ chạy lại ô này, không mất
công đã làm.


In [ ]:
# [1.6] CHẠY ĐẦY ĐỦ
if "EDA_OUT" not in globals():
    raise SystemExit("Chưa chạy ô [1.1] CẤU HÌNH. "
                     "Hãy chạy lần lượt từ ô [1.1] trở xuống rồi thử lại.")

lenh_day_du = [sys.executable, "scripts/audit.py",
               "--config", "configs/data.yaml",
               "--output-dir", EDA_OUT]
print("Chạy: " + " ".join(lenh_day_du))
print("=" * 72)
kq = subprocess.run(lenh_day_du, cwd=REPO_DIR, stderr=subprocess.STDOUT)
print("=" * 72)
print("Mã thoát:", kq.returncode, "(0 là thành công)")

In [ ]:
# [1.7] XEM BÁO CÁO
if "EDA_OUT" not in globals():
    raise SystemExit("Chưa chạy ô [1.1] CẤU HÌNH. "
                     "Hãy chạy lần lượt từ ô [1.1] trở xuống rồi thử lại.")

import os

duong_bao_cao = f"{EDA_OUT}/EDA_REPORT.md"
print(open(duong_bao_cao, encoding="utf-8").read())

thu_muc_overlay = f"{EDA_OUT}/overlay"
ds_anh = sorted(os.listdir(thu_muc_overlay)) if os.path.isdir(thu_muc_overlay) else []
print(f"\nẢnh overlay ({len(ds_anh)} tệp) trong {thu_muc_overlay}:")
for t in ds_anh:
    print("  " + t)

---
## Gửi kết quả cho Claude

Copy 3 thứ này:

1. **Toàn bộ phần `CẤU TRÚC ZIP`** của ô [1.5] (kể cả các dòng `[!]` cảnh báo).
2. **Nội dung `EDA_REPORT.md`** ở ô [1.7].
3. **Nhận xét bằng mắt** sau khi mở thư mục `overlay` trên Drive:
   - Box có **khớp với nhà** không, hay lệch/bao gồm cả đường, cây?
   - Có thấy **nhiều nhà bị gộp thành một box** không? (box viền **tím** là những
     box to bất thường — nghi bị gộp)
   - Nhà trong ảnh có **quá nhỏ** không? (để trả lời câu hỏi có phải cắt tile)

Nhận xét bằng mắt quan trọng ngang số liệu: máy chỉ đo được diện tích và số
component, còn "box này có đúng là một căn nhà không" thì chỉ người nhìn mới biết.
